# Bhashantor v2: Chakma / Marma / Bangla / Banglish / English (NLLB fine-tune)

**আগে করো:** Runtime → Change runtime type → **T4 GPU** → Save। তারপর উপর থেকে নিচে এক এক cell চালাও।

- `QUICK_TEST = True` → ~১০ মিনিটের smoke test। কাজ করলে `False` করে আবার প্রথম থেকে চালাও।
- ভাষা-কোড: `en` English, `bn` Bangla, `bl` Banglish, `ccp` Chakma, `rmz` Marma
- Training file তৈরি করতে (কম্পিউটারে): `python prepare_data.py --train-per-direction 4000`

In [ ]:
# 1) GPU check + install
import torch
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change runtime type!")
!pip install -q transformers datasets accelerate sacrebleu sentencepiece

In [ ]:
# 2) Settings + Drive (checkpoints are saved to Drive, survive a disconnect)
MODEL_NAME = "facebook/nllb-200-distilled-600M"
QUICK_TEST = True        # True = small smoke test. Set False for the real run.
MAX_LEN    = 64
EPOCHS     = 1 if QUICK_TEST else 2
LR         = 1e-4
USE_FP16   = True        # if loss shows nan, set to False
NEW_TAGS   = ["<2en>", "<2bn>", "<2bl>", "<2ccp>", "<2rmz>"]

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/bhashantor/ckpt" if not QUICK_TEST else "/content/quick_ckpt"
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

# chrF++ on the test set to beat (from your own runs):
# RETRIEVAL = nearest training sentence ; LEXICON = word-by-word dictionary (lexicon_translate.py eval)
RETRIEVAL = {"en-bn": 36.45, "en-bl": 38.89, "en-ccp": 22.46, "en-rmz": 21.59, "bn-en": 36.39, "bn-bl": 52.35, "bn-ccp": 23.95, "bn-rmz": 21.18, "bl-en": 35.68, "bl-bn": 50.32, "bl-ccp": 23.72, "bl-rmz": 22.24, "ccp-en": 25.68, "ccp-bn": 27.26, "ccp-bl": 30.27, "ccp-rmz": 20.17, "rmz-en": 21.76, "rmz-bn": 21.26, "rmz-bl": 24.57, "rmz-ccp": 17.81}
LEXICON   = {'en-bn': 31.94, 'en-bl': 36.54, 'en-ccp': 24.44, 'en-rmz': 27.55, 'bn-en': 30.69, 'bn-bl': 83.77, 'bn-ccp': 31.42, 'bn-rmz': 30.14, 'bl-en': 30.03, 'bl-bn': 81.24, 'bl-ccp': 32.04, 'bl-rmz': 30.61, 'ccp-en': 20.41, 'ccp-bn': 29.9, 'ccp-bl': 33.68, 'ccp-rmz': 24.24, 'rmz-en': 17.23, 'rmz-bn': 20.83, 'rmz-bl': 24.58, 'rmz-ccp': 18.98}

### ৩) Data upload
এই **৩টা** file দাও (`ai-model/data/processed/` থেকে): `train_all_directions.jsonl`, `valid_all_directions.jsonl`, `test_all_directions.jsonl`

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 3b) if names came as "xxx (1).jsonl", fix them
import os, glob
for kind in ["train", "valid", "test"]:
    dup = sorted(glob.glob(f"{kind}_all_directions (*).jsonl"), key=os.path.getmtime)
    if dup:
        os.replace(dup[-1], f"{kind}_all_directions.jsonl")
print({f: os.path.getsize(f) for f in glob.glob("*_all_directions*.jsonl")})

In [ ]:
# 4) Load data
from datasets import load_dataset
raw = load_dataset("json", data_files={
    "train": "train_all_directions.jsonl",
    "validation": "valid_all_directions.jsonl",
    "test": "test_all_directions.jsonl"})
if QUICK_TEST:
    raw["train"] = raw["train"].shuffle(seed=42).select(range(3000))
    raw["validation"] = raw["validation"].shuffle(seed=42).select(range(500))
else:
    raw["validation"] = raw["validation"].shuffle(seed=42).select(range(2000))
print(raw)
print(raw["train"][0])

In [ ]:
# 5) Tokenizer + model (direction tags as new special tokens)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang="eng_Latn", tgt_lang="eng_Latn")
tok.add_tokens(NEW_TAGS, special_tokens=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tok))
with torch.no_grad():
    emb = model.get_input_embeddings().weight
    emb[-len(NEW_TAGS):] = emb[:-len(NEW_TAGS)].mean(dim=0, keepdim=True)
print("vocab:", len(tok), "| params (M):", round(model.num_parameters()/1e6))

def prep(batch):
    return tok(batch["source"], text_target=batch["target"], max_length=MAX_LEN, truncation=True)
tokenized = raw.map(prep, batched=True, remove_columns=raw["train"].column_names)
print(tokenized)

In [ ]:
# 6) Train (auto-resumes from the last Drive checkpoint)
import math, glob
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

BS, ACCUM = 16, 2
steps_per_epoch = math.ceil(len(tokenized["train"]) / (BS * ACCUM))
total_steps = steps_per_epoch * EPOCHS
warmup = max(1, int(0.03 * total_steps))
print("steps/epoch:", steps_per_epoch, "| total:", total_steps, "| warmup:", warmup)

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BS, per_device_eval_batch_size=32,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR, num_train_epochs=EPOCHS, warmup_steps=warmup,
    optim="adafactor", fp16=USE_FP16, group_by_length=True,
    eval_strategy="steps", eval_steps=1000 if not QUICK_TEST else 50,
    save_strategy="steps", save_steps=1000 if not QUICK_TEST else 50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=100 if not QUICK_TEST else 25, report_to="none")
trainer = Seq2SeqTrainer(model=model, args=args,
    train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForSeq2Seq(tok, model=model),
    processing_class=tok)
has_ckpt = len(glob.glob(f"{OUT_DIR}/checkpoint-*")) > 0
trainer.train(resume_from_checkpoint=True if has_ckpt else None)
trainer.save_model(f"{OUT_DIR}/final_model"); tok.save_pretrained(f"{OUT_DIR}/final_model")

In [ ]:
# 7) Translate helper (any direction: en, bn, bl, ccp, rmz)
BOS = tok.convert_tokens_to_ids("eng_Latn")

def translate(texts, tgt, bs=64, beams=4):
    model.eval(); outs = []
    for i in range(0, len(texts), bs):
        batch = [f"<2{tgt}> {t}" for t in texts[i:i+bs]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=BOS, num_beams=beams,
                                 max_new_tokens=MAX_LEN, no_repeat_ngram_size=3)
        outs += tok.batch_decode(gen, skip_special_tokens=True)
    return outs

print(translate(["তুমি কোথায় থাকো?", "আমি তোমাকে ভালোবাসি।"], "ccp"))
print(translate(["tumi kothay thako?", "ami tomake bhalobashi"], "ccp"))
print(translate(["Where do you live?"], "ccp"), translate(["Where do you live?"], "rmz"))

In [ ]:
# 8) Evaluate on TEST. Sentences and names are scored SEPARATELY (names are easy copies and would inflate scores)
import pandas as pd, sacrebleu, re
test = raw["test"].to_pandas()
if "kind" not in test.columns:
    test["kind"] = "sentence"
norm = lambda s: re.sub(r"[^\w]", "", s.lower())
rows, all_preds = [], []
for (d, kind), g in test.groupby(["direction", "kind"]):
    if QUICK_TEST: g = g.head(200 if kind == "sentence" else 60)
    tgt = d.split("-")[1]
    preds = translate(g["source_plain"].tolist(), tgt)
    if tgt in ("bl", "ccp", "rmz"):                      # references are lowercase by design
        preds = [p.lower() for p in preds]
    refs = g["target"].tolist()
    chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=2).score
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    exact = 100 * sum(norm(p) == norm(r) for p, r in zip(preds, refs)) / len(refs)
    row = {"kind": kind, "direction": d, "model_chrF++": round(chrf, 2), "exact_match_%": round(exact, 1)}
    if kind == "sentence":
        best = max(RETRIEVAL.get(d, 0), LEXICON.get(d, 0))
        row.update({"retrieval": RETRIEVAL.get(d), "lexicon": LEXICON.get(d), "gain_vs_best": round(chrf - best, 2),
                    "model_BLEU": round(bleu, 2)})
    rows.append(row)
    all_preds.append(pd.DataFrame({"kind": kind, "direction": d, "source": g["source_plain"].values, "prediction": preds, "reference": refs}))
    print(row)
res = pd.DataFrame(rows).sort_values(["kind", "direction"])
res.to_csv(f"{OUT_DIR}/model_scores.csv" if not QUICK_TEST else "model_scores.csv", index=False)
pd.concat(all_preds).to_csv("test_predictions.csv", index=False)
res
